# Mini Assignment 1 — Lung Cancer Patient Health and Treatment Records

## Outline
0. Project Summary
1. Setup
2. Load Dataset
3. Task 1 — Data Cleaning
4. Task 2 — Treatment Duration Analysis
5. Task 3 — Highest Survival Rate by Smoking Status
6. Task 4 — Top 3 Countries by Stage IV Percentage
7. Task 5 — Patient Filtering and Analysis
8. Assumptions

#Project Summary

Mini Assignment 1 — Lung Cancer Patient Health and Treatment Records

This project uses PySpark to perform data cleaning, transformation, aggregation, filtering, and analysis on the Lung Cancer Patient Health and Treatment Records dataset.
The analysis consists of five main tasks:

1.	Data Cleaning:
Removed duplicate records, ensured appropriate data types for numerical and date columns, and converted Yes/No categorical fields into a binary 1/0 representation.

2.	Treatment Duration Analysis:
Created a treatment_duration_days column by calculating the number of days between diagnosis and the end of treatment. The average treatment duration was then calculated for each treatment type.

3.	Smoking Status & Survival:
Calculated the survival rate for each smoking-status group and identified the group with the highest survival rate.

4.	Stage IV Analysis by Country:
Calculated the percentage of patients diagnosed with Stage IV cancer for each country and identified the top three countries with the highest percentage.

5.	Patient Filtering & Health Analysis:
Filtered patients based on gender, cancer stage, family history, smoking status, BMI, and survival status. For the resulting group, the average age and percentage of patients with hypertension were calculated.


# Setup

In [1]:
# Install PySpark
!pip install pyspark -q

# Import PySpark
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("Mini_Assignment_1_Lung_Cancer") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark session created successfully!")

Spark version: 4.0.4
Spark session created successfully!


# Loading Dataset

In [5]:
from google.colab import files

uploaded = files.upload()

Saving Lung Cancer.csv to Lung Cancer.csv


# Task 1 — Data Cleaning

In [6]:
# Load the Lung Cancer dataset
df = spark.read.csv(
    "/content/Lung Cancer.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [7]:
# Inspect the dataset
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: string (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)



In [8]:
# Display the first 5 rows
df.show(5, truncate=False)

+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|id |age |gender|country    |diagnosis_date|cancer_stage|family_history|smoking_status|bmi |cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|1  |64.0|Male  |Sweden     |2016-04-05    |Stage I     |Yes           |Passive Smoker|29.4|199              |0           |0     |1        |0           |Chemotherapy  |2017-09-10        |0       |
|2  |50.0|Female|Netherlands|2023-04-20    |Stage III   |Yes           |Passive Smoker|41.2|280              |1           |1     |0        |0           |Surgery       |2024-06-17        |1       |
|3  |65.0|Femal

In [9]:
from pyspark.sql.functions import col, when, lower, trim


def clean_data(df):

    # 1. Remove duplicate rows
    df = df.dropDuplicates()

    # 2. Ensure correct numerical data types
    df = df.withColumn("id", col("id").cast("integer"))
    df = df.withColumn("age", col("age").cast("double"))
    df = df.withColumn("bmi", col("bmi").cast("double"))
    df = df.withColumn("cholesterol_level", col("cholesterol_level").cast("integer"))
    df = df.withColumn("hypertension", col("hypertension").cast("integer"))
    df = df.withColumn("asthma", col("asthma").cast("integer"))
    df = df.withColumn("cirrhosis", col("cirrhosis").cast("integer"))
    df = df.withColumn("other_cancer", col("other_cancer").cast("integer"))
    df = df.withColumn("survived", col("survived").cast("integer"))

    # 3. Ensure correct date data types
    df = df.withColumn("diagnosis_date", col("diagnosis_date").cast("date"))
    df = df.withColumn("end_treatment_date", col("end_treatment_date").cast("date"))

    # 4. Convert Yes/No field to 1/0
    df = df.withColumn(
        "family_history",
        when(lower(trim(col("family_history"))) == "yes", 1)
        .when(lower(trim(col("family_history"))) == "no", 0)
        .otherwise(None)
        .cast("integer")
    )

    return df

In [10]:
clean_df = clean_data(df)

print("Data cleaning completed successfully!")

Data cleaning completed successfully!


In [11]:
clean_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: integer (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)



In [12]:
clean_df.show(5, truncate=False)

+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|id |age |gender|country       |diagnosis_date|cancer_stage|family_history|smoking_status|bmi |cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|72 |54.0|Female|Czech Republic|2017-06-30    |Stage IV    |0             |Passive Smoker|24.0|160              |1           |0     |0        |1           |Surgery       |2019-04-06        |0       |
|190|34.0|Male  |Estonia       |2015-02-13    |Stage I     |1             |Former Smoker |38.8|252              |1           |1     |1        |0           |Combined      |2016-12-18        |0       |


In [13]:
# Check for duplicate rows after cleaning
total_rows = clean_df.count()
distinct_rows = clean_df.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)

if total_rows == distinct_rows:
    print("No duplicate rows remain.")
else:
    print("Duplicate rows still exist.")

Total rows: 890000
Distinct rows: 890000
No duplicate rows remain.


# Task 2 — Treatment Duration Analysis

In [14]:
from pyspark.sql.functions import datediff

duration_df = clean_df.withColumn(
    "treatment_duration_days",
    datediff(col("end_treatment_date"), col("diagnosis_date"))
)

duration_df.show(5, truncate=False)

+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
|id |age |gender|country       |diagnosis_date|cancer_stage|family_history|smoking_status|bmi |cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|treatment_duration_days|
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
|72 |54.0|Female|Czech Republic|2017-06-30    |Stage IV    |0             |Passive Smoker|24.0|160              |1           |0     |0        |1           |Surgery       |2019-04-06        |0       |645                    |
|190|34.0|Male  |Estonia       |2015-02-13    |Stage I     |1             |Former Smoker |38.8|252      

In [16]:
from pyspark.sql.functions import avg

avg_duration_df = duration_df.groupBy("treatment_type").agg(
    avg("treatment_duration_days").alias("average_treatment_duration_days")
)

avg_duration_df.show()

+--------------+-------------------------------+
|treatment_type|average_treatment_duration_days|
+--------------+-------------------------------+
|     Radiation|             458.40320462900917|
|  Chemotherapy|             458.39540091909953|
|      Combined|              457.8152186120058|
|       Surgery|             457.73744630723684|
+--------------+-------------------------------+



# Task 3 — Highest Survival Rate by Smoking Status

In [17]:
from pyspark.sql.functions import avg, desc

highest_first_df = clean_df.groupBy("smoking_status").agg(
    avg("survived").alias("survival_rate")
)

highest_first_df = highest_first_df.orderBy(
    desc("survival_rate")
)

highest_first_df.show()

+--------------+-------------------+
|smoking_status|      survival_rate|
+--------------+-------------------+
|  Never Smoked|0.22091034383684025|
|Current Smoker| 0.2203399760250205|
|Passive Smoker| 0.2200250929784469|
| Former Smoker|0.21964074335789288|
+--------------+-------------------+



### Result

The smoking-status group with the highest survival rate is **Never Smoked**, with a survival rate of approximately **22.09%**.

# Task 4 — Top 3 Countries by Stage IV Percentage

In [18]:
##IS STAGE IV##

from pyspark.sql.functions import when

stage4_df = clean_df.withColumn(
    "is_stage_iv",
    when(col("cancer_stage") == "Stage IV", 1).otherwise(0)
)

stage4_df.show(5, truncate=False)

+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------+
|id |age |gender|country       |diagnosis_date|cancer_stage|family_history|smoking_status|bmi |cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|is_stage_iv|
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------+
|72 |54.0|Female|Czech Republic|2017-06-30    |Stage IV    |0             |Passive Smoker|24.0|160              |1           |0     |0        |1           |Surgery       |2019-04-06        |0       |1          |
|190|34.0|Male  |Estonia       |2015-02-13    |Stage I     |1             |Former Smoker |38.8|252              |1           |1     |1        |0        

In [19]:
from pyspark.sql.functions import avg

stage4_percentage_df = stage4_df.groupBy("country").agg(
    (avg("is_stage_iv") * 100).alias("stage_iv_percentage")
)

stage4_percentage_df.show()

+--------------+-------------------+
|       country|stage_iv_percentage|
+--------------+-------------------+
|        Sweden|  24.92988751847049|
|       Germany| 24.666059502125076|
|        France| 25.214614898039095|
|        Greece|  25.50223889628464|
|      Slovakia| 24.637019450278512|
|       Belgium|  24.74383071606136|
|       Finland| 24.876516860784196|
|         Malta| 25.135613030838854|
|       Croatia| 25.427002233085883|
|         Italy| 25.203350734490716|
|     Lithuania|  25.12478694911127|
|         Spain| 25.107439017008655|
|       Denmark|  25.11809593023256|
|       Ireland|  24.90749932316578|
|        Cyprus| 25.236101347840705|
|       Estonia| 24.309123521721947|
|        Latvia|  25.07293106095501|
|Czech Republic| 25.291166185190818|
|      Slovenia| 25.147704893198004|
|    Luxembourg|  24.80636701835702|
+--------------+-------------------+
only showing top 20 rows


#Top three Countries

In [20]:
from pyspark.sql.functions import desc

top_3_countries = stage4_percentage_df.orderBy(
    desc("stage_iv_percentage")
).limit(3)

top_3_countries.show()

+--------------+-------------------+
|       country|stage_iv_percentage|
+--------------+-------------------+
|        Greece|  25.50223889628464|
|       Croatia| 25.427002233085883|
|Czech Republic| 25.291166185190818|
+--------------+-------------------+



#Task 5 — Patient Filtering and Analysis

In [21]:
filtered_df = clean_df.filter(
    (col("gender") == "Male") &
    ((col("cancer_stage") == "Stage III") | (col("cancer_stage") == "Stage IV")) &
    (col("family_history") == 1) &
    (col("smoking_status") == "Current Smoker") &
    (col("bmi") > 30) &
    (col("survived") == 1)
)

filtered_df.show()

+------+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|    id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+------+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| 18230|63.0|  Male|   Bulgaria|    2023-02-21|   Stage III|             1|Current Smoker|32.7|              268|           1|     0|        0|           0|     Radiation|        2023-10-01|       1|
| 82700|63.0|  Male|Netherlands|    2021-10-01|    Stage IV|             1|Current Smoker|30.8|              245|           1|     1|        1|           0|     Radiation|        2023-02-08|       1|


In [22]:
from pyspark.sql.functions import avg

task5_result = filtered_df.agg(
    avg("age").alias("average_age"),
    avg("hypertension").alias("hypertension_percentage")
)

task5_result = task5_result.withColumn(
    "hypertension_percentage",
    col("hypertension_percentage") * 100
)

task5_result.show()

+------------------+-----------------------+
|       average_age|hypertension_percentage|
+------------------+-----------------------+
|55.179398872886665|      74.76518472135254|
+------------------+-----------------------+



#Assumptions

Assumptions Made

•	Duplicate rows were considered records with identical values across all columns and were removed.

•	Numerical columns such as age, bmi, and cholesterol_level were treated as numeric values.

•	diagnosis_date and end_treatment_date were treated as date columns.

•	Yes/No fields were converted to 1/0, where Yes = 1 and No = 0.

•	The survived column was assumed to use 1 = survived and 0 = did not survive.

•	The family_history column was interpreted as 1 = Yes and 0 = No after conversion.

•	For survival-rate calculations, the average of the binary survived column was used; therefore, avg(survived) × 100 represents the survival percentage.

•	For Stage IV analysis, the percentage was calculated as the average of a binary Stage IV indicator multiplied by 100.

•	For the hypertension analysis, the average of the binary hypertension column multiplied by 100 represents the percentage of filtered patients with hypertension.

•	Treatment duration was calculated as end_treatment_date - diagnosis_date in days.

•	Records with missing or invalid values were handled according to the transformations applied during data cleaning.

•	The dataset is assumed to be synthetic/non-real patient data, as specified in the assignment instructions.




